In [5]:
from pathlib import Path

import simulation as sim
import xobjects as xo

## Build line from lattice file

In [6]:
line = sim.build_line(folder=Path('Lattice_Files/02_Aperture_Lattice/'))

Converting sequence "synchrotron":   0%|          | 0/1990 [00:00<?, ?it/s]

In [ ]:
x = [0]
y = [0]
px = [0]
py = [0]

tw = line.twiss(method='4d', freeze_longitudinal=True)
particles = line.build_particles(
    x=x+tw.x[0],
    y=y+tw.y[0],
    px=px+tw.px[0],
    py=py+tw.py[0],
    method='4d', freeze_longitudinal=True) # idk if this line is necessary

# For a matched / gaussian beam
gemitt = 50e-6
"""
ref = line.particle_ref
bg = ref.beta0 * ref.gamma0

particles = line.build_particles(
    x_norm=np.random.normal(size=n_particles),
    y_norm=np.random.normal(size=n_particles),
    px_norm=np.random.normal(size=n_particles),
    py_norm=np.random.normal(size=n_particles),
    nemitt_x=bg*gemitt,
    nemitt_y=bg*gemitt,
    method='4d',
    freeze_longitudinal=True, mode='normalized_transverse')

# Ensures distribution is truly centered on closed orbit
particles.x += (tw.x[0] - np.mean(particles.x))
particles.y += (tw.y[0] - np.mean(particles.y))
particles.px += (tw.px[0] - np.mean(particles.px))
particles.py += (tw.py[0] - np.mean(particles.py))
"""

## Add monitors
Call twiss after monitors so monitors are in twiss table however you cannot call twiss after my custom SC elements

In [ ]:
line = sim.add_monitors(line, particles, n_monitors=1, n_turns=n_turns)
twiss = line.twiss(method='4d', freeze_longitudinal=True)

# Add spacecharge elements
The outputs are dataframes to be passed onto save you you can recover beam and aperture information in analysis

In [ ]:
line, beam_df, aper_df = sim.add_spacecharge(line, 
                                             beam_intensity=5e12, # Number of protons 
                                             n_interactions=100, # Number of space charge interactions
                                             gemitt=gemitt, # Geometric emittance (use nemitt for normalised)
                                                            # Can also input `[e_x, e_y]` if they arent the same
                                             indirect=True, # Is the indirect SC used
                                             pipe_enlargement=1, # multiplier to make the pipe bigger (useful for my report)
                                                                 # changing the pipesize also changes the boundary 
                                                                 # for losing particles
                                             xfields=False # To use the xfields element instead of my own (this is direct only)
                                                           # you can verify it gives the same results 
                                                           # as my own with indirect=False
                                            )

In [ ]:
line.build_tracker(_context=xo.ContextCpu(omp_num_threads=0))
line.track(particles, num_turns=400, time=True)

In [ ]:
sim.save(line, 
         particles, 
         twiss, 
         filename=f'example', # Saves as `example.h5`
         aper_df=aper_df, 
         beam_df=beam_df)